# Face Recognition Model Training
Training face recognition model using LFW People dataset

In [ ]:
import numpy as np
import pandas as pd
import cv2
import os
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
import matplotlib.pyplot as plt

## Download Dataset
Download from: https://www.kaggle.com/datasets/atulanandjha/lfwpeople

In [ ]:
# Load and preprocess LFW dataset
dataset_path = 'datasets/lfw-people'

def load_images(path, img_size=(160, 160)):
    images = []
    labels = []
    
    for person_name in os.listdir(path):
        person_path = os.path.join(path, person_name)
        if not os.path.isdir(person_path):
            continue
            
        for img_name in os.listdir(person_path):
            img_path = os.path.join(person_path, img_name)
            img = cv2.imread(img_path)
            img = cv2.resize(img, img_size)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            images.append(img)
            labels.append(person_name)
    
    return np.array(images), np.array(labels)

# Load data
X, y = load_images(dataset_path)
X = X / 255.0  # Normalize

print(f"Dataset shape: {X.shape}")
print(f"Number of unique people: {len(np.unique(y))}")

In [ ]:
# Build FaceNet-inspired model
def build_facenet_model(input_shape=(160, 160, 3), embedding_size=128):
    inputs = Input(shape=input_shape)
    
    x = Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = MaxPooling2D((2, 2))(x)
    
    x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2))(x)
    
    x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2))(x)
    
    x = Flatten()(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.5)(x)
    embeddings = Dense(embedding_size, activation='linear', name='embeddings')(x)
    
    model = Model(inputs=inputs, outputs=embeddings)
    return model

model = build_facenet_model()
model.summary()

In [ ]:
# Save model
model.save('../models/face_recognition_model.h5')
print("Model saved successfully!")